# vlm-evaluation-harness quickstart

Runs fully offline against the built-in mock adapter — no API keys, no network access, no GPU. This mirrors the README quick start, one cell per step, so you can inspect the actual return values.

Install first if you haven't: `pip install -e .` from the repo root.

In [ ]:
import subprocess


def run(*args):
    proc = subprocess.run(["vlm-evaluation-harness", *args], capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc

## 1. Discriminative eval: image + text → text, scored against ground truth

In [ ]:
_ = run("eval", "--model", "mock:demo-v1", "--bench", "demo_mc")

## 2. Generative eval: text → image, scored by an LLM-judge

In [ ]:
_ = run("gen-eval", "--model", "mock:t2i-v1", "--bench", "genjudge_mini")

## 3. Compare two models on the same benchmark

In [ ]:
_ = run("compare", "--models", "mock:demo-v1,mock:demo-v2", "--bench", "demo_mc")

## 4. Regression tracking: is model B actually worse than model A?

Every `eval`/`gen-eval` run above was appended to `~/.vlm-evaluation-harness/history.jsonl` with per-sample scores. `regression` runs a paired McNemar test over the two latest tracked runs — severity is driven by statistical significance, not a fixed percentage delta.

In [ ]:
_ = run("history")
_ = run("regression", "--baseline", "mock:demo-v1", "--current", "mock:demo-v2")

## 5. Aggregate saved runs into one HTML/Markdown report

In [ ]:
import os
import tempfile

out_dir = tempfile.mkdtemp()
_ = run("eval", "--model", "mock:demo-v1", "--bench", "demo_mc", "--output-dir", out_dir)
report_path = os.path.join(out_dir, "report.html")
_ = run("report", "--results-dir", out_dir, "--output", report_path)
print("report written to", report_path)

## Next steps

- Swap `mock:demo-v1` for a real backend: `pip install -e ".[anthropic]"` then `--model anthropic:claude-opus-4-6`.
- See `docs/adding-a-benchmark.md` to point this at your own data.
- See `docs/adding-an-adapter.md` to wire up a model backend that isn't built in.
- See `docs/reproducibility.md` for what's recorded in every result and how `reproduce` replays a run.